In [1]:
import pandas as pd
import glob
import os
import requests
import rasterio
import numpy as np
import json

In [ ]:


# 1. Căutăm fișierele (presupunem că sunt în folderul 'Dataseturi copaci')
cale_dataseturi = os.path.join('..', 'Dataseturi copaci')
fisiere_gasite = glob.glob(os.path.join(cale_dataseturi, "*.csv"))

mapare_coloane = {
    'Categorii de fructe': 'Specie',
    'Categorii de pomi fructiferi': 'Specie',
    'Macroregiuni  regiuni de dezvoltare si judete': 'Judet',
    'Ani': 'An'
}

lista_df = []

for fisier in fisiere_gasite:
    df = pd.read_csv(fisier)
    df.columns = df.columns.str.strip()
    df.rename(columns=mapare_coloane, inplace=True)
    
    # Standardizăm numele speciilor imediat după citire
    # Schimbăm "Meri" în "Mere" ca să se potrivească toate fișierele
    if 'Specie' in df.columns:
        df['Specie'] = df['Specie'].str.strip().replace('Meri', 'Mere')
    
    # Filtrăm DOAR pentru "Mere" chiar acum (economisim memorie)
    df = df[df['Specie'] == 'Mere']
    
    nume = os.path.basename(fisier).lower()
    if 'nr_pomi' in nume:
        df.rename(columns={'Valoare': 'Numar_Pomi'}, inplace=True)
    elif 'kg_pom' in nume:
        df.rename(columns={'Valoare': 'Kg_pe_Pom'}, inplace=True)
    elif 'tone_totale' in nume:
        df.rename(columns={'Valoare': 'Tone_Totale'}, inplace=True)

    # Convertim valorile la numere
    for col in ['Numar_Pomi', 'Kg_pe_Pom', 'Tone_Totale']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    cols_prezente = [c for c in ['Specie', 'Judet', 'An', 'Numar_Pomi', 'Kg_pe_Pom', 'Tone_Totale'] if c in df.columns]
    lista_df.append(df[cols_prezente])

if lista_df:
    # Unim datele filtrate pentru Meri
    df_merge = pd.concat(lista_df, ignore_index=True)
    
    # Consolidăm pe rânduri unice (Specie, Judet, An)
    # Folosim sum() pentru a colecta datele din fișiere diferite pe același rând
    df_final = df_merge.groupby(['Specie', 'Judet', 'An'], as_index=False).agg({
        'Numar_Pomi': lambda x: x.sum(min_count=1),
        'Kg_pe_Pom': lambda x: x.sum(min_count=1),
        'Tone_Totale': lambda x: x.sum(min_count=1)
    })


    print(f"Am extras datele pentru {len(df_final)} înregistrări cu Meri/Mere.")
    print(df_final.head())
    
    # Salvare rezultat
    df_final.to_csv('date_meri_final.csv', index=False)

Am extras datele pentru 1072 înregistrări cu Meri/Mere.
  Specie  Judet          An  Numar_Pomi  Kg_pe_Pom  Tone_Totale
0   Mere   Alba   Anul 2000    874496.0        8.0       6759.0
1   Mere   Alba   Anul 2001    828878.0       12.0       9728.0
2   Mere   Alba   Anul 2002    829665.0       11.0       8752.0
3   Mere   Alba   Anul 2003    828750.0       17.0      13811.0
4   Mere   Alba   Anul 2004    856253.0       36.0      30537.0


In [21]:
df_copaci = pd.read_csv('date_meri_final.csv')
df_sol = pd.read_csv('sol_judete_romania.csv')

df_sol['Judet'] = df_sol['Judet'].str.strip()
df_copaci['Judet'] = df_copaci['Judet'].str.strip()

df_sol = df_sol[['Judet', 'phh2o', 'clay', 'silt', 'sand', 'soc', 'nitrogen', 'cec' ]]

df_final = pd.merge(df_copaci, df_sol, on='Judet', how='inner')

df_final.head()



,Specie,Judet,An,Numar_Pomi,Kg_pe_Pom,Tone_Totale,phh2o,clay,silt,sand,soc,nitrogen,cec
0,Mere,Alba,Anul 2000,874496.0,8.0,6759.0,6.283,33.189,37.161,29.672,34.439,3.087,25.511
1,Mere,Alba,Anul 2001,828878.0,12.0,9728.0,6.283,33.189,37.161,29.672,34.439,3.087,25.511
2,Mere,Alba,Anul 2002,829665.0,11.0,8752.0,6.283,33.189,37.161,29.672,34.439,3.087,25.511
3,Mere,Alba,Anul 2003,828750.0,17.0,13811.0,6.283,33.189,37.161,29.672,34.439,3.087,25.511
4,Mere,Alba,Anul 2004,856253.0,36.0,30537.0,6.283,33.189,37.161,29.672,34.439,3.087,25.511


In [3]:
df_master['Max_Kg_Istoric'] = df_master.groupby('Judet')['Kg_per_Pom'].transform('max')

# 2. Modificăm funcția să se uite la acest potențial maxim
def identifica_rootstock_stabil(row):
    potenial_maxim = row['Max_Kg_Istoric']
    
    if potenial_maxim < 20:   # Pragurile pot fi ajustate
        return 1  # Pitic (M9)
    elif potenial_maxim < 38:
        return 2  # Semipitic (M26)
    elif potenial_maxim < 65:
        return 3  # Semiviguros (MM106)
    else:
        return 4  # Viguros (Salbatic)

# 3. Aplicăm pe coloana de potențial
df_master['Rootstock_ID'] = df_master.apply(identifica_rootstock_stabil, axis=1)

print(df_master.head())

           An   Judet Specie_Pomi  Nr_Pomi  Tone_Totale  Kg_per_Pom  \
0   Anul 2000   Bihor       Pruni  1563587        15430          10   
1   Anul 2000   Bihor       Pruni  1563587        15430           7   
2   Anul 2000   Bihor       Pruni  1563587        15430           9   
3   Anul 2000   Bihor       Pruni  1563587        15430           9   
4   Anul 2000   Bihor       Pruni  1563587        15430           8   

   Max_Kg_Istoric  Rootstock_ID  
0              45             3  
1              45             3  
2              45             3  
3              45             3  
4              45             3  
